In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Imports successful!")

Imports successful!


In [2]:
PROJECT_ROOT = Path("..")

CSV_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / "landmarks.csv"
)

print("CSV path:", CSV_PATH.resolve())
print("Exists:", CSV_PATH.exists())

CSV path: D:\Code FIles\VS code files\Project\sign-language-translator\dataset\processed\landmarks.csv
Exists: True


In [3]:
df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (74137, 64)


,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,x18,y18,z18,x19,y19,z19,x20,y20,z20,label
0,0.455066,0.583699,-6.535667e-07,0.569869,0.507308,-0.034267,0.639892,0.374764,-0.041253,0.651905,...,0.391801,0.323449,-0.071389,0.404981,0.408848,-0.056393,0.403411,0.461151,-0.029737,A
1,0.484675,0.614162,-8.084496e-07,0.602145,0.543912,-0.025812,0.676195,0.407449,-0.030664,0.689200,...,0.433379,0.362128,-0.071392,0.446727,0.449884,-0.059598,0.444696,0.504641,-0.038163,A
2,0.562319,0.796830,-6.815258e-07,0.706193,0.709073,-0.047565,0.801895,0.552039,-0.058396,0.821115,...,0.456043,0.469220,-0.094205,0.467616,0.581387,-0.079842,0.475096,0.657608,-0.047797,A
3,0.723497,0.674151,-6.063991e-07,0.796600,0.617823,-0.031191,0.847890,0.513129,-0.036425,0.862495,...,0.623255,0.453557,-0.038356,0.630417,0.509658,-0.031472,0.640656,0.555732,-0.014557,A
4,0.713202,0.752712,-4.978478e-07,0.830028,0.651184,-0.042247,0.897470,0.498115,-0.055674,0.895898,...,0.604023,0.449111,-0.108978,0.632722,0.545297,-0.094283,0.638129,0.612068,-0.063292,A


In [4]:
print(df.columns.tolist())
print("\nNumber of columns:", len(df.columns))

['x0', 'y0', 'z0', 'x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4', 'x5', 'y5', 'z5', 'x6', 'y6', 'z6', 'x7', 'y7', 'z7', 'x8', 'y8', 'z8', 'x9', 'y9', 'z9', 'x10', 'y10', 'z10', 'x11', 'y11', 'z11', 'x12', 'y12', 'z12', 'x13', 'y13', 'z13', 'x14', 'y14', 'z14', 'x15', 'y15', 'z15', 'x16', 'y16', 'z16', 'x17', 'y17', 'z17', 'x18', 'y18', 'z18', 'x19', 'y19', 'z19', 'x20', 'y20', 'z20', 'label']

Number of columns: 64


In [5]:
def normalize_landmarks(row):
    # Convert 63 features into:
    # 21 landmarks × 3 coordinates
    landmarks = row.iloc[:63].to_numpy(
        dtype=np.float64
    ).reshape(21, 3)

    # ---------------------------------
    # 1. Translation normalization
    # ---------------------------------
    # Landmark 0 = wrist
    wrist = landmarks[0].copy()

    landmarks = landmarks - wrist

    # ---------------------------------
    # 2. Scale normalization
    # ---------------------------------
    # Find the largest 3D distance
    # from wrist to any landmark
    distances = np.linalg.norm(
        landmarks,
        axis=1
    )

    scale = distances.max()

    # Avoid division by zero
    if scale > 1e-8:
        landmarks = landmarks / scale

    # Return to 63 features
    return landmarks.flatten()

In [6]:
feature_columns = [
    col for col in df.columns
    if col != "label"
]

normalized_data = np.vstack(
    df.apply(
        normalize_landmarks,
        axis=1
    ).values
)

normalized_df = pd.DataFrame(
    normalized_data,
    columns=feature_columns
)

normalized_df["label"] = df["label"].values

print("Original shape  :", df.shape)
print("Normalized shape:", normalized_df.shape)

display(normalized_df.head())

Original shape  : (74137, 64)
Normalized shape: (74137, 64)


,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,x18,y18,z18,x19,y19,z19,x20,y20,z20,label
0,0.0,0.0,0.0,0.257292,-0.171206,-0.076796,0.414228,-0.468262,-0.092453,0.441151,...,-0.141788,-0.583268,-0.159994,-0.112251,-0.391872,-0.126385,-0.115770,-0.274653,-0.066645,A
1,0.0,0.0,0.0,0.260283,-0.155655,-0.057191,0.424360,-0.458023,-0.067943,0.453175,...,-0.113659,-0.558441,-0.158185,-0.084083,-0.363996,-0.132052,-0.088583,-0.242669,-0.084557,A
2,0.0,0.0,0.0,0.250643,-0.152880,-0.082862,0.417366,-0.426450,-0.101731,0.450850,...,-0.185143,-0.570728,-0.164113,-0.164982,-0.375322,-0.139092,-0.151951,-0.242538,-0.083266,A
3,0.0,0.0,0.0,0.210664,-0.162323,-0.089884,0.358470,-0.464027,-0.104965,0.400557,...,-0.288874,-0.635696,-0.110532,-0.268234,-0.474029,-0.090692,-0.238727,-0.341255,-0.041949,A
4,0.0,0.0,0.0,0.223575,-0.194299,-0.080848,0.352640,-0.487231,-0.106544,0.349632,...,-0.208939,-0.581012,-0.208554,-0.154017,-0.396938,-0.180431,-0.143669,-0.269156,-0.121124,A


In [7]:
print("Missing values:")
print(normalized_df.isnull().sum().sum())

print("\nInfinite values:")
print(
    np.isinf(
        normalized_df[feature_columns].to_numpy()
    ).sum()
)

print("\nNumber of samples:", len(normalized_df))
print("Number of classes:", normalized_df["label"].nunique())

print("\nClass distribution:")
print(
    normalized_df["label"]
    .value_counts()
    .sort_index()
)

Missing values:
0

Infinite values:
0

Number of samples: 74137
Number of classes: 29

Class distribution:
label
A          2617
B          2770
C          2622
D          2903
E          2522
F          2998
G          2832
H          2844
I          2668
J          2912
K          2820
L          2849
M          2279
N          1944
O          2642
P          2474
Q          2528
R          2771
S          2765
T          2601
U          2774
V          2759
W          2681
X          2357
Y          2720
Z          2673
del        2287
nothing      36
space      2489
Name: count, dtype: int64


In [8]:
sample = normalized_df.iloc[0]

print("Wrist after normalization:")
print(
    "x0 =", sample["x0"],
    "y0 =", sample["y0"],
    "z0 =", sample["z0"]
)

coords = sample[feature_columns].to_numpy(
    dtype=np.float64
).reshape(21, 3)

distances = np.linalg.norm(coords, axis=1)

print("\nMaximum landmark distance:")
print(distances.max())

Wrist after normalization:
x0 = 0.0 y0 = 0.0 z0 = 0.0

Maximum landmark distance:
0.9999999999999999


In [9]:
X = normalized_df.drop(columns=["label"])
y = normalized_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", y.nunique())

X shape: (74137, 63)
y shape: (74137,)
Classes: 29


In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder_v2 = LabelEncoder()

y_encoded = label_encoder_v2.fit_transform(y)

print("Number of encoded classes:", len(label_encoder_v2.classes_))
print("Classes:")
print(label_encoder_v2.classes_)

Number of encoded classes: 29
Classes:
['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z' 'del' 'nothing' 'space']


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 59309
Testing samples : 14828


In [12]:
from sklearn.preprocessing import StandardScaler

scaler_v2 = StandardScaler()

X_train_scaled = scaler_v2.fit_transform(X_train)
X_test_scaled = scaler_v2.transform(X_test)

print("Scaling completed!")
print("Train shape:", X_train_scaled.shape)
print("Test shape :", X_test_scaled.shape)

Scaling completed!
Train shape: (59309, 63)
Test shape : (14828, 63)


In [13]:
### SVM V2
from sklearn.svm import SVC

svm_v2 = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

print("Training SVM V2...")

svm_v2.fit(
    X_train_scaled,
    y_train
)

print("SVM V2 training completed!")

Training SVM V2...
SVM V2 training completed!


In [14]:
y_pred_v2 = svm_v2.predict(X_test_scaled)

print("Predictions completed!")

Predictions completed!


In [15]:
from sklearn.metrics import accuracy_score

accuracy_v2 = accuracy_score(
    y_test,
    y_pred_v2
)

print(
    f"Normalized SVM V2 Accuracy: "
    f"{accuracy_v2 * 100:.2f}%"
)

Normalized SVM V2 Accuracy: 98.84%


In [16]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_v2,
        target_names=label_encoder_v2.classes_
    )
)

              precision    recall  f1-score   support

           A       0.98      0.97      0.98       523
           B       0.99      0.99      0.99       554
           C       1.00      1.00      1.00       524
           D       0.99      0.99      0.99       581
           E       1.00      0.99      1.00       504
           F       1.00      0.99      1.00       600
           G       1.00      1.00      1.00       566
           H       0.99      0.99      0.99       569
           I       0.99      0.99      0.99       534
           J       0.99      1.00      0.99       583
           K       0.99      0.99      0.99       564
           L       1.00      0.99      1.00       570
           M       0.95      0.98      0.96       456
           N       0.96      0.94      0.95       389
           O       0.99      0.99      0.99       528
           P       0.99      0.99      0.99       495
           Q       0.98      0.99      0.99       506
           R       0.99    

In [17]:
import joblib
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    svm_v2,
    MODEL_DIR / "svm_model_v2.pkl"
)

joblib.dump(
    scaler_v2,
    MODEL_DIR / "scaler_v2.pkl"
)

joblib.dump(
    label_encoder_v2,
    MODEL_DIR / "label_encoder_v2.pkl"
)

print("V2 model files saved successfully!")

V2 model files saved successfully!
